# ASG Airlines End-to-End Data Engineering Project
## Step 4: Data Quality Checks & Data Governance Audit

---

### 1. Objective
The primary objective of **Step 4: Data Quality Checks** is to perform a rigorous data quality audit on the ingested ASG Airlines dataset across all 4 raw Excel sheets (`flights`, `payments`, `bookings`, `passengers`).

**Core Mandates:**
- Identify schema anomalies, missing values, duplicates, invalid formats, and data corruptions.
- Catalog Personally Identifiable Information (PII) columns for compliance.
- Output a structured Data Quality Report (`data/processed/data_quality_report.csv`).
- **Immutability Guarantee:** No data cleaning or modifications are made to the raw dataset in this step. All recommendations are prepared for **Step 5: Data Cleaning & Transformation**.

### 2. Dataset Overview
We load the raw Excel workbook from `data/raw/UseCase - Airlines.xlsx` using `pandas` and `openpyxl`.

In [ ]:
import os
import pandas as pd
import numpy as np

RAW_DATA_PATH = os.path.join("..", "data", "raw", "UseCase - Airlines.xlsx")
excel_file = pd.ExcelFile(RAW_DATA_PATH, engine="openpyxl")

sheets = {sheet: pd.read_excel(excel_file, sheet_name=sheet) for sheet in excel_file.sheet_names}
for name, df in sheets.items():
    print(f"Sheet: {name:<12} | Rows: {df.shape[0]:<6} | Columns: {df.shape[1]:<4}")

### 3. Schema Checks
Inspecting sheet names, column names, data types, and checking for missing/unexpected columns across sheets.

In [ ]:
for sheet_name, df in sheets.items():
    print(f"\n--- {sheet_name.upper()} SCHEMA ---")
    for col, dtype in df.dtypes.items():
        print(f"  - {col}: {dtype}")

### 4. Missing Value Analysis
Calculating missing value counts and percentage missing per column across all sheets.

In [ ]:
missing_summary = []
for sheet_name, df in sheets.items():
    for col in df.columns:
        null_cnt = df[col].isnull().sum()
        if null_cnt > 0:
            pct = round((null_cnt / len(df)) * 100, 2)
            missing_summary.append({"Sheet": sheet_name, "Column": col, "Missing Count": null_cnt, "Missing %": pct})

missing_df = pd.DataFrame(missing_summary)
display(missing_df)

### 5. Duplicate Analysis
Checking for exact full-row duplicates and duplicate primary keys (`flight_id`, `passenger_id`, etc.).

In [ ]:
dup_summary = []
for sheet_name, df in sheets.items():
    full_dups = df.duplicated().sum()
    dup_summary.append({"Sheet": sheet_name, "Total Rows": len(df), "Full Duplicate Rows": full_dups})

display(pd.DataFrame(dup_summary))

# Detailed duplicate flight records
df_flights = sheets["flights"]
dup_flight_ids = df_flights[df_flights.duplicated(subset=["flight_id"], keep=False)]
print(f"Total Duplicate flight_id records: {len(dup_flight_ids)}")
display(dup_flight_ids.sort_values("flight_id").head(6))

### 6. Flight ID Validation
Validating `flight_id` values against standard regex format rule (`^[A-Z0-9]{2}\d{3}$`) and flagging conflicting records.

In [ ]:
flight_id_pattern = r"^[A-Z0-9]{2}\d{3}$"
invalid_fids = df_flights[~df_flights["flight_id"].astype(str).str.match(flight_id_pattern, na=False)]
print(f"Null Flight IDs: {df_flights['flight_id'].isnull().sum()}")
print(f"Malformed Flight IDs (Rule: ^[A-Z0-9]{{2}}\d{{3}}$): {len(invalid_fids)}")

### 7. Time Validation
Inspecting `departure_time`, `arrival_time`, and `duration` for unparseable formats or cross-day anomalies (where `arrival_time < departure_time`).

In [ ]:
negative_durations = df_flights[df_flights["arrival_time"] < df_flights["departure_time"]]
print(f"Overnight/Cross-day Flights with Arrival < Departure: {len(negative_durations)}")
display(negative_durations[["flight_id", "departure_time", "arrival_time", "duration"]])

### 8. Categorical Consistency
Checking unique categorical values in `airline`, `status`, `payment_method`, `source`, and `destination` for invalid strings or casing inconsistencies.

In [ ]:
print("Flights Airline Values:", df_flights["airline"].value_counts(dropna=False).to_dict())
print("Bookings Status Values:", sheets["bookings"]["status"].value_counts(dropna=False).to_dict())
print("Payments Payment Method:", sheets["payments"]["payment_method"].value_counts(dropna=False).to_dict())

### 9. PII Identification
Cataloging columns containing sensitive Personally Identifiable Information (PII) for compliance and masking.

In [ ]:
pii_catalog = [
    {"Sheet": "passengers", "Column": "first_name", "PII Category": "Direct PII", "Sensitivity": "High"},
    {"Sheet": "passengers", "Column": "last_name", "PII Category": "Direct PII", "Sensitivity": "High"},
    {"Sheet": "passengers", "Column": "email", "PII Category": "Direct PII / Contact", "Sensitivity": "High"},
    {"Sheet": "passengers", "Column": "phone", "PII Category": "Direct PII / Contact", "Sensitivity": "High"},
    {"Sheet": "passengers", "Column": "aadhaar_id", "PII Category": "Government ID", "Sensitivity": "Critical"},
    {"Sheet": "passengers", "Column": "date_of_birth", "PII Category": "Demographic PII", "Sensitivity": "Medium"},
    {"Sheet": "bookings", "Column": "passport_number", "PII Category": "Government ID", "Sensitivity": "Critical"},
    {"Sheet": "bookings", "Column": "emergency_contact_name", "PII Category": "Third-Party PII", "Sensitivity": "Medium"},
    {"Sheet": "bookings", "Column": "emergency_contact_phone", "PII Category": "Third-Party Contact", "Sensitivity": "Medium"}
]
display(pd.DataFrame(pii_catalog))

### 10. Data Quality Summary
We load and view the generated `data_quality_report.csv` file created by `src/data_quality.py`.

In [ ]:
report_path = os.path.join("..", "data", "processed", "data_quality_report.csv")
report_df = pd.read_csv(report_path)
display(report_df)

### 11. Recommended Cleaning Actions for Step 5
Based on our quality audit, the following actions will be executed in **Step 5: Data Cleaning & Transformation**:

1. **Deduplication:** Remove 15 exact full-row duplicates in `flights` and resolve duplicate `passenger_id`s in `passengers`.
2. **Missing Value Imputation:**
   - Impute missing `airline` values in `flights` using `flight_id` prefix lookup (e.g., prefix `6F` -> `IndiGo`).
   - Impute missing `last_name` in `passengers` with `'N/A'` or empty string.
   - Impute missing/invalid `amount` in `payments` using median amount by payment method.
   - Impute missing/invalid `status` in `bookings` as `'UNKNOWN'` or `'PENDING'`.
3. **Data Type Coercion:** Coerce `'INVALID'` string values in `payments['amount']` to numeric and `bookings['status']` to standard categories.
4. **Timestamp & Date Fixing:** Resolve arrival timestamp overflow for flight `SJ192`.
5. **Aadhaar ID Formatting:** Left-pad truncated numeric `aadhaar_id` entries with leading zeros to standard 12-digit format.
6. **Anonymization & Governance:** Apply hashing or masking to PII columns (`email`, `phone`, `aadhaar_id`, `passport_number`) for production environment compliance.